# El Parser: De Texto Web a Tensores Numéricos

## 1. Contexto y Preparación: 
Son el equipo de Inteligencia Artificial de una firma de análisis móvil. Necesitan entrenar un modelo para predecir el éxito de una app, pero el archivo CSV que les mandó el equipo de Scraping es un desastre de formato. Tienen que convertirlo en una matriz matemática estricta.

## 2. Instrucciones Técnicas: 
Creen un Notebook  nuevo y carga el dataset crudo:

In [119]:
import pandas as pd
import numpy as np
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/googleplaystore.csv"
df = pd.read_csv(url)

## 3. Preguntas de Análisis (Para responder con código y celdas Markdown):

### Pregunta A: 
El scraper cometió errores y guardó la misma app varias veces. Escriban una línea de código para identificar si hay nombres duplicados en la columna App (df.duplicated(subset=['App']).sum()). Si los hay, apliquen .drop_duplicates(subset=['App'], keep='first', inplace=True) para quedarse solo con la primera aparición de cada aplicación. ¿Cuántos registros duplicados purgaron?

In [120]:
total_apps = df["App"].count()
print("total:", total_apps)

duplicated_apps = df.duplicated(subset=["App"]).sum()
print("duplicates:", duplicated_apps)

# df_cleaned_A = df.copy()
# df_cleaned_A.drop_duplicates(subset=["App"],keep="first",inplace=True)
df.drop_duplicates(subset=["App"],keep="first",inplace=True)


app_count_no_duplicates = df["App"].count()
print("new total:", app_count_no_duplicates)

if (total_apps - app_count_no_duplicates == duplicated_apps):
    print("Duplicated apps removed:", duplicated_apps)


total: 10841
duplicates: 1181
new total: 9660
Duplicated apps removed: 1181


### Pregunta B: 
La columna Installs tiene texto como "1,000,000+". Un algoritmo no puede procesar eso.

a) Escriban una instrucción encadenando el método .str.replace() para eliminar el símbolo + y otro para eliminar la coma ,. (Ojo: utilicen regex=False o escapen el símbolo más \+).

b) Al final de la cadena, utilicen .astype(int) (o float si tienen NaNs) para convertir la columna.

c) Comprueben el éxito ejecutando df['Installs'].mean(). ¿Cuál es el promedio real de descargas?

In [121]:
# df["Installs"].mean() - fails

# df["Installs"] = df["Installs"].str.replace(r"[+,]","",regex=True)
#df["Installs"] = pd.to_numeric(df["Installs"],errors="coerce")

df["Installs"] = df["Installs"].str.replace("+","",regex=False)
df["Installs"] = df["Installs"].str.replace(",","",regex=False)
df["Installs"] = df["Installs"].str.replace("Free","0",regex=False)
df["Installs"] = df["Installs"].astype(float)


print("promedio: ",df["Installs"].mean())

# run this cell with run all so no problems on the .str.replace lines

promedio:  7776701.607349897


### Pregunta C: 
La columna Price tiene valores como "\$4.99" y "0". Apliquen la misma lógica arquitectónica que en la Pregunta B: eliminen el símbolo de dólar \$ y conviertan la columna usando .astype(float).

In [122]:
df["Price"] = df["Price"].str.replace("$","", regex=False)
# df["Price"] = df["Price"].str.replace("Everyone","0", regex=False)

df["Price"] = pd.to_numeric(df["Price"],errors="coerce")


print("promedio: ",df["Price"].mean())


promedio:  1.0992990992856404


### Pregunta D: 
Ahora que el precio es numérico, ejecuten df['Price'].max(). Notarán que la app más cara cuesta alrededor de 400 dólares.

a) Usen un filtro booleano (df[df['Price'] > 200]) para descubrir cuáles son estas apps. Verán que son aplicaciones basura o de broma.

In [123]:
df["Price"].max()

400.0

In [124]:
df[df["Price"]>200]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
4197,most expensive app (H),FAMILY,4.3,6,1.5M,100.0,Paid,399.99,Everyone,Entertainment,"July 16, 2018",1.0,7.0 and up
4362,💎 I'm rich,LIFESTYLE,3.8,718,26M,10000.0,Paid,399.99,Everyone,Lifestyle,"March 11, 2018",1.0.0,4.4 and up
4367,I'm Rich - Trump Edition,LIFESTYLE,3.6,275,7.3M,10000.0,Paid,400.00,Everyone,Lifestyle,"May 3, 2018",1.0.1,4.1 and up
5351,I am rich,LIFESTYLE,3.8,3547,1.8M,100000.0,Paid,399.99,Everyone,Lifestyle,"January 12, 2018",2.0,4.0.3 and up
5354,I am Rich Plus,FAMILY,4.0,856,8.7M,10000.0,Paid,399.99,Everyone,Entertainment,"May 19, 2018",3.0,4.4 and up
5355,I am rich VIP,LIFESTYLE,3.8,411,2.6M,10000.0,Paid,299.99,Everyone,Lifestyle,"July 21, 2018",1.1.1,4.3 and up
5356,I Am Rich Premium,FINANCE,4.1,1867,4.7M,50000.0,Paid,399.99,Everyone,Finance,"November 12, 2017",1.6,4.0 and up
5357,I am extremely Rich,LIFESTYLE,2.9,41,2.9M,1000.0,Paid,379.99,Everyone,Lifestyle,"July 1, 2018",1.0,4.0 and up
5358,I am Rich!,FINANCE,3.8,93,22M,1000.0,Paid,399.99,Everyone,Finance,"December 11, 2017",1.0,4.1 and up
5359,I am rich(premium),FINANCE,3.5,472,965k,5000.0,Paid,399.99,Everyone,Finance,"May 1, 2017",3.4,4.4 and up


b) Si metemos estas apps al modelo, la red neuronal creerá que vender a 400 dólares es un modelo de negocio viable. Sobrescriban el DataFrame conservando únicamente las apps que cuesten menos de 50 dólares (df = df[df['Price'] < 50]).

In [125]:
df= df[df["Price"]<50]

In [126]:
df[df["Price"]>200]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver


c) Guarden su dataset final usando df.to_csv('playstore_limpio.csv', index=False).


In [ ]:
df.to_csv("playstore_limpio.csv",index=False)